In [1]:
import pandas as pd
import ray
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
import plotly.graph_objects as go
from itertools import combinations
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import (train_test_split, GridSearchCV,
                                    StratifiedKFold, StratifiedShuffleSplit,
                                    cross_val_score)
from sklearn.metrics import make_scorer, recall_score, precision_score, f1_score, classification_report
from imblearn.ensemble import BalancedRandomForestClassifier
import numpy as np
import joblib
from pandarallel import pandarallel
from sklearn.preprocessing import LabelEncoder, StandardScaler
import re

2025-05-26 16:03:37.984190: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-26 16:03:37.986182: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-26 16:03:38.018849: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-05-26 16:03:38.018876: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-05-26 16:03:38.018897: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to regi

In [2]:
def busqueda(expression,column,name=None):
    if column is np.nan:
        return np.nan
    m = re.search(expression,column)
    if m == None:
        return np.nan
    if name != None:
        return name
    else :
        return m.group(0)

In [52]:
df_2 = pd.read_csv("/home/nicolas/nico/Data/Masivas/data_correccion_zari/ZariDR3_2arcscSkiff.csv")
df_2 = df_2.reset_index()

In [53]:
lista = pd.DataFrame()

In [54]:
skiff = df_2[['index','source_id',"skiff_type",'Bibcode','GroupID_skiff']].astype(str)

In [55]:
skiff['skiff_type'] = skiff['skiff_type'].replace('nan', np.nan)
skiff['Bibcode'] = skiff['Bibcode'].replace('nan', np.nan)
skiff['GroupID_skiff'] = skiff['GroupID_skiff'].replace('nan', np.nan)

In [56]:
skiff = skiff.loc[skiff['skiff_type'].notna()]
print(f"{len(skiff)} Estrellas no unicas con informacion espectral en skiff")

123133 Estrellas no unicas con informacion espectral en skiff


In [57]:
lista["sp_skiff"] = [len(skiff)]

In [58]:
resultado = skiff.apply(lambda row: busqueda(r":|\?", row["skiff_type"]), axis=1)
sp_peculiaridades = skiff.loc[resultado.notna()]["skiff_type"].unique()
print(f"Estrellas con uncertanity in their sp \n {sp_peculiaridades}")
skiff = skiff.loc[resultado.isna()]
lista["uncertanity"] = [len(resultado.loc[resultado.notna()])]

Estrellas con uncertanity in their sp 
 ['A3:' 'O9III:' 'B1:e' ... 'O9.5I?p' 'B3IV:n' 'O7.5IV((f)) + O9III:']


In [59]:
#resultado  = skiff.apply(lambda row: busqueda("\+",row["skiff_type"]),axis=1)
#print(f"{len(resultado.loc[resultado.notna()])} Distarted Binaries stars")
#skiff = skiff.loc[resultado.isna()].reset_index(drop=True)
#lista["binaries"] = [len(resultado.loc[resultado.notna()])]

In [60]:
resultado  = skiff.apply(lambda row: busqueda("\+",row["skiff_type"]),axis=1)
print(f"{len(resultado.loc[resultado.notna()])} Distarted Binaries stars")
skiff = skiff.loc[resultado.notna()].reset_index(drop=True)

2110 Distarted Binaries stars


In [61]:
resultado_mas = skiff.apply(lambda row: busqueda(r'\+$',row["skiff_type"]),axis=1)
resultado_OB = skiff.apply(lambda row: busqueda(r'OB',row["skiff_type"]),axis=1)
resultado_f = skiff.apply(lambda row: busqueda(r'f+',row["skiff_type"]),axis=1)
resultado_sd = skiff.apply(lambda row: busqueda(r'sd',row["skiff_type"]),axis=1)
resultado_em = skiff.apply(lambda row: busqueda(r'em',row["skiff_type"]),axis=1)
resultado_obe = skiff.apply(lambda row: busqueda(r"OBe", row["skiff_type"]),axis=1)
resultado_ob = skiff.apply(lambda row: busqueda(r"OB", row["skiff_type"]),axis=1)
resultado_w = skiff.apply(lambda row: busqueda(r"W", row["skiff_type"]),axis=1)
resultado_be = skiff.apply(lambda row: busqueda(r"Be", row["skiff_type"]),axis=1)
resultado_peq = skiff.apply(lambda row: busqueda(r"(e|em|H|w|wl|wk|v|k|ak|Sr|Si|sh|shell|h|p|pec|s|n|Cr|nn|He|m|Fe|Ca|Mg|Na|Ti|CN|:|\?)",row["skiff_type"]),axis=1)
resultado_peq2 = skiff.apply(lambda row: busqueda(r"(N|C|PN|R|DA|S|Q|L)",row["skiff_type"]),axis=1)

In [62]:
skiff = skiff.loc[
    (resultado_mas.isna()) &
    (resultado_OB.isna()) &
    (resultado_f.isna()) &
    (resultado_sd.isna()) &
    (resultado_em.isna()) &
    (resultado_obe.isna()) &
    (resultado_ob.isna()) &
    (resultado_w.isna()) &
    (resultado_be.isna()) &
    (resultado_peq.isna()) &
    (resultado_peq2.isna())
]

split_cols = skiff["skiff_type"].str.split(r'\+', expand=True)
split_cols = split_cols.iloc[:, :2]  # Asegura solo 2 columnas
split_cols.columns = ["sp1", "sp2"]
skiff[["sp1", "sp2"]] = split_cols


In [63]:

# Function to generate the new type
def generate_type(type_str):
    types = type_str.split('/')
    new_types = []
    
    for t in types:
        primary_type = t
        subtype_value = np.random.randint(0,9)
        new_types.append(f"{primary_type}{subtype_value}")
    
    return '/'.join(new_types)

In [64]:
def spectral_to_number(spectral_type):
    letter = spectral_type[0]
    number = spectral_type[1:].replace('/', '.')
    base_number = replace_map[letter]
    return f"{base_number}.{number}"

In [65]:
def convert_to_number(value):
    match = re.match(r'([OBAFGKM])(\d?)/([OBAFGKM])(\d?)$', value)
    if match:
        letter1, num1, letter2, num2 = match.groups()
        
        num1 = num1 if num1 else str(np.random.randint(0, 9))
        num2 = num2 if num2 else str(np.random.randint(0, 9))
        
        return f"{letter1}{num1}/{letter2}{num2}"
    return None

In [66]:
resultado = skiff.apply(lambda row: busqueda(r'[OBAFKGM]\d(?:\.\d)?/[OBAFKGM]\d(?:\.\d)?', row["sp2"]), axis=1)
skiff.loc[resultado.notna(),"mk2"] = resultado

resultado = skiff.apply(lambda row: busqueda(r'[OBAFKGM]\d(?:\.\d)?/[OBAFKGM]\d(?:\.\d)?', row["sp1"]), axis=1)
skiff.loc[resultado.notna(),"mk1"] = resultado

In [67]:
skiff

,index,source_id,skiff_type,Bibcode,GroupID_skiff,sp1,sp2,mk2,mk1
1,36,1869256701670871168,B0 + B0,1965ApJ...141..955K,12.0,B0,B0,NaN,NaN
2,37,1869256701670871168,O9.8V + O9.8V,1968ApJ...153..187O,12.0,O9.8V,O9.8V,NaN,NaN
3,38,1869256701670871168,B0V + B0V,1971ApJ...170..325C,12.0,B0V,B0V,NaN,NaN
4,41,1869256701670871168,O9.3 + O9.4,1995A&A...297..127H,12.0,O9.3,O9.4,NaN,NaN
5,42,1869256701670871168,O9V + O9.5V,1997ApJ...490..328B,12.0,O9V,O9.5V,NaN,NaN
...,...,...,...,...,...,...,...,...,...
2062,1087642,4146599682284818176,O6.5V + O8V,2009MNRAS.400.1479S,25265.0,O6.5V,O8V,NaN,NaN
2071,1088128,4146600781797073920,O7V + B0.5V + B0.5V,2009MNRAS.400.1479S,25359.0,O7V,B0.5V,NaN,NaN
2091,1089838,2204369910027981568,B3V + B5V,1968ApJ...154..923S,25692.0,B3V,B5V,NaN,NaN
2092,1089901,3276605300008567296,B3(V) + A7(V),1980ApJS...44..489B,25701.0,B3(V),A7(V),NaN,NaN


In [68]:
resultado = skiff.apply(lambda row: busqueda(r'[OBAFGKM]/[A-Za-z]\d$|[A-Za-z]\d/[OBAFGKM]$', row["sp2"]), axis=1)
skiff.loc[resultado.notna(), 'mk2'] = skiff.loc[resultado.notna(), 'sp2'].apply(convert_to_number)


resultado = skiff.apply(lambda row: busqueda(r'[OBAFGKM]/[A-Za-z]\d$|[A-Za-z]\d/[OBAFGKM]$', row["sp1"]), axis=1)
skiff.loc[resultado.notna(), 'mk1'] = skiff.loc[resultado.notna(), 'sp1'].apply(convert_to_number)


In [69]:
skiff

,index,source_id,skiff_type,Bibcode,GroupID_skiff,sp1,sp2,mk2,mk1
1,36,1869256701670871168,B0 + B0,1965ApJ...141..955K,12.0,B0,B0,NaN,NaN
2,37,1869256701670871168,O9.8V + O9.8V,1968ApJ...153..187O,12.0,O9.8V,O9.8V,NaN,NaN
3,38,1869256701670871168,B0V + B0V,1971ApJ...170..325C,12.0,B0V,B0V,NaN,NaN
4,41,1869256701670871168,O9.3 + O9.4,1995A&A...297..127H,12.0,O9.3,O9.4,NaN,NaN
5,42,1869256701670871168,O9V + O9.5V,1997ApJ...490..328B,12.0,O9V,O9.5V,NaN,NaN
...,...,...,...,...,...,...,...,...,...
2062,1087642,4146599682284818176,O6.5V + O8V,2009MNRAS.400.1479S,25265.0,O6.5V,O8V,NaN,NaN
2071,1088128,4146600781797073920,O7V + B0.5V + B0.5V,2009MNRAS.400.1479S,25359.0,O7V,B0.5V,NaN,NaN
2091,1089838,2204369910027981568,B3V + B5V,1968ApJ...154..923S,25692.0,B3V,B5V,NaN,NaN
2092,1089901,3276605300008567296,B3(V) + A7(V),1980ApJS...44..489B,25701.0,B3(V),A7(V),NaN,NaN


In [70]:
resultado = skiff.apply(lambda row: busqueda(r'[OBAFGKM]\d', row["sp2"]), axis=1)
skiff.loc[(resultado.notna())&(skiff["mk2"].isna()),"mk2"] = resultado.loc[skiff["mk2"].isna()]

resultado = skiff.apply(lambda row: busqueda(r'[OBAFGKM]\d', row["sp1"]), axis=1)
skiff.loc[(resultado.notna())&(skiff["mk1"].isna()),"mk1"] = resultado.loc[skiff["mk1"].isna()]


In [71]:
skiff

,index,source_id,skiff_type,Bibcode,GroupID_skiff,sp1,sp2,mk2,mk1
1,36,1869256701670871168,B0 + B0,1965ApJ...141..955K,12.0,B0,B0,B0,B0
2,37,1869256701670871168,O9.8V + O9.8V,1968ApJ...153..187O,12.0,O9.8V,O9.8V,O9,O9
3,38,1869256701670871168,B0V + B0V,1971ApJ...170..325C,12.0,B0V,B0V,B0,B0
4,41,1869256701670871168,O9.3 + O9.4,1995A&A...297..127H,12.0,O9.3,O9.4,O9,O9
5,42,1869256701670871168,O9V + O9.5V,1997ApJ...490..328B,12.0,O9V,O9.5V,O9,O9
...,...,...,...,...,...,...,...,...,...
2062,1087642,4146599682284818176,O6.5V + O8V,2009MNRAS.400.1479S,25265.0,O6.5V,O8V,O8,O6
2071,1088128,4146600781797073920,O7V + B0.5V + B0.5V,2009MNRAS.400.1479S,25359.0,O7V,B0.5V,B0,O7
2091,1089838,2204369910027981568,B3V + B5V,1968ApJ...154..923S,25692.0,B3V,B5V,B5,B3
2092,1089901,3276605300008567296,B3(V) + A7(V),1980ApJS...44..489B,25701.0,B3(V),A7(V),A7,B3


In [77]:
skiff.loc[skiff["mk2"].isna(), "sp2"].str.replace(" ","").str[0]

208     G
347     B
386     B
387     B
481     B
540     F
693     B
714     B
715     B
747     A
773     B
783     B
804     B
874     O
925     I
1016    G
1062    A
1190    B
1192    A
1240    q
1318    O
1525    B
1616    A
1636    A
1660    B
1732    A
1757    B
1787    A
1842    A
1867    G
1869    B
1871    B
1946    F
1999    G
Name: sp2, dtype: object

In [78]:
skiff.loc[skiff["mk2"].isna(), "mk2"] = (
    skiff.loc[skiff["mk2"].isna(), "sp2"].str.replace(" ","").str[0] + 
    np.random.randint(0, 9, skiff["mk2"].isna().sum()).astype(str)
)


skiff.loc[skiff["mk1"].isna(), "mk1"] = (
    skiff.loc[skiff["mk1"].isna(), "sp1"].str.replace(" ","").str[0] + 
    np.random.randint(0, 9, skiff["mk1"].isna().sum()).astype(str)
)

In [79]:
skiff['mk1'] = skiff['mk1'].str.replace('.', '')

skiff['mk2'] = skiff['mk2'].str.replace('.', '')

In [80]:
replace_map = {
    'O': '0.',
    'B': '1.',
    'A': '2.',
    'F': '3.',
    'G': '4.',
    'K': '5.',
    'M': '6.'
}

skiff['mk2'] = skiff['mk2'].replace(replace_map, regex=True)

skiff['mk1'] = skiff['mk1'].replace(replace_map, regex=True)

In [91]:
skiff = skiff.loc[(skiff["mk2"]!="I2")&(skiff["mk2"]!='q4')]

In [93]:
def promedio_redondeado(valor):
    if '/' in valor:
        numeros = [float(n) for n in valor.split('/')]
        return np.random.uniform(numeros[0],numeros[1])
    else:
        return valor


        
# Aplica la función a la columna 'mk'
skiff['mk2'] = skiff['mk2'].apply(promedio_redondeado).astype(float)

skiff['mk1'] = skiff['mk1'].apply(promedio_redondeado).astype(float)

/tmp/ipykernel_29708/535796976.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  skiff['mk2'] = skiff['mk2'].apply(promedio_redondeado).astype(float)
/tmp/ipykernel_29708/535796976.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  skiff['mk1'] = skiff['mk1'].apply(promedio_redondeado).astype(float)


In [95]:
import pandas as pd

def process_skiff_dataframe(skiff,label, std_threshold,coded):
    # Filtra los datos en función de la columna "GroupID_skiff"
    stars_unicas = skiff.loc[skiff["GroupID_skiff"].isna()].reset_index(drop=True)
    star_no_unicas = skiff.loc[skiff["GroupID_skiff"].notna()].reset_index(drop=True)

    # Agrupa por "GroupID_skiff" y obtiene los tipos únicos de "skiff_type"
    star_no_unicas = star_no_unicas.merge(
        star_no_unicas.groupby("GroupID_skiff").agg({
            "Bibcode": lambda x: list(x.unique()),
            "index": lambda x: list(x.unique()),
            "skiff_type": lambda x: list(x.unique()),
            "source_id": lambda x: list(x.unique())
        }).reset_index(),
        on="GroupID_skiff",
        suffixes=('', '_list')
    )

    # Calcula la desviación estándar por grupo y filtra los datos según el umbral
    std_by_group = star_no_unicas.groupby("GroupID_skiff")[coded].transform("std")
    star_no_unicas["std_sp"] = std_by_group
    star_no_unicas = star_no_unicas.loc[(star_no_unicas["std_sp"] < std_threshold) | (star_no_unicas["std_sp"].isna())]

    # Calcula la media por grupo y la asigna como etiqueta
    mean_by_group = star_no_unicas.groupby("GroupID_skiff")[coded].transform("mean")
    star_no_unicas[label] = mean_by_group

    # Elimina duplicados
    star_no_unicas = star_no_unicas.drop_duplicates(subset="GroupID_skiff")

    # Asigna etiquetas y lista de tipos para estrellas únicas
    stars_unicas[label] = stars_unicas[coded]
    stars_unicas["skiff_type_list"] = stars_unicas["skiff_type"]

    # Combina ambos DataFrames
    skiff_mk = pd.concat([star_no_unicas, stars_unicas])

    return skiff_mk


In [96]:
skiff_1 = skiff.copy()
skiff_2 = skiff.copy()

In [99]:
skiff_mk_1 = process_skiff_dataframe(skiff_1,"label_1", 0.3,"mk1")


In [100]:
skiff_mk_2 = process_skiff_dataframe(skiff_2,"label_2", 0.3,"mk2")

In [102]:
skiff_mk_2 = skiff_mk_2.rename(columns={
                          "std_sp":"std_sp_2"})

In [103]:
skiff_mk_1 = skiff_mk_1.merge(skiff_mk_2[["index","label_2","std_sp_2"]],how="inner")

In [107]:
skiff_mk_1["label_1"] = np.round(skiff_mk_1["label_1"],1)
skiff_mk_1["label_2"] = np.round(skiff_mk_1["label_2"],1)

In [110]:
skiff_mk_1 =skiff_mk_1.drop(columns="label")

In [111]:
skiff_mk_1.to_csv("/home/nicolas/nico/Data/Masivas/Data_OB_stars/binaries_codeed.csv")

In [216]:
skiff_mk = skiff_mk.rename(columns={"label":"label1"})

In [47]:
skiff_mk.to_csv("/home/nicolas/nico/Data/Masivas/Data_OB_stars/skiff_2arcsec_OBAGDR3_coded_V2.csv",index=False)

In [63]:
skiff_mk["source_id"] = skiff_mk["source_id"].astype(int)
skiff_mk["index"] = skiff_mk["index"].astype(int)
skiff_mk["GroupID_skiff"] = skiff_mk["GroupID_skiff"].astype(float)

In [68]:
df_2 = df_2.merge(skiff_mk[['index', 'source_id', 'skiff_type', 'Bibcode', 'GroupID_skiff', 'mk',
       'Bibcode_list', 'index_list', 'skiff_type_list', 'source_id_list',
       'std_sp', 'label']], on=['source_id','index','skiff_type','GroupID_skiff','Bibcode'], how='left')

In [ ]:
df_2.to_csv("/home/nicolas/nico/Data/Masivas/Data_OB_stars/skiff_2arcsec_OBAGDR3_prep_V2.csv",
           index=False)

In [472]:
df_3 = pd.read_csv("/home/nicolas/nico/Data/Masivas/Data_OB_stars/skiff_2arcsec_OBAGDR3_prep_V2.csv")

In [475]:
estrellas_con_al_menos_otra_class = len(uniques_OB.loc[uniques_OB["GroupID_skiff"].notna()])

In [477]:
estrellas_con_al_menos_otra_class/ len(uniques_OB)

0.7189761215629522

In [360]:
OB_not_coded = df_2.loc[(df_2["GroupID_skiff"].notna())&(df_2["skiff_type"]=="OB")]

In [361]:
coded = df_3.loc[(df_3["mk"].notna())&(df_3["GroupID_skiff"].notna())]

In [362]:
coded = coded[["mk","GroupID_skiff"]].merge(OB_not_coded,how="inner",on="GroupID_skiff")

In [450]:
uniques_OB_coded = len(coded.drop_duplicates(subset="source_id"))

In [452]:
(uniques_OB_coded/uniques_OB)*100

42.94500723589002